In [30]:
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

data = pd.read_csv(r'C:\Users\menda\Downloads\dataset_2.csv', header=None)
print("Эхний 5 бичлэг:")
print(data.head())

Эхний 5 бичлэг:
                               0
0             MILK,BREAD,BISCUIT
1  BREAD,MILK,BISCUIT,CORNFLAKES
2            BREAD,TEA,BOURNVITA
3           JAM,MAGGI,BREAD,MILK
4              MAGGI,TEA,BISCUIT


In [31]:
transactions = []
for i in range(data.shape[0]):
    row_str = str(data.iloc[i, 0])
    items = [item.strip() for item in row_str.split(',')]
    transactions.append(items)
print("\nЭхний 3 гүйлгээний жагсаалт:")
print(transactions[:3])


Эхний 3 гүйлгээний жагсаалт:
[['MILK', 'BREAD', 'BISCUIT'], ['BREAD', 'MILK', 'BISCUIT', 'CORNFLAKES'], ['BREAD', 'TEA', 'BOURNVITA']]


In [32]:
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

print("\n--- One-hot кодлогдсон хүснэгт (эхний 10 мөр) ---")
print(df_encoded.head(10))


--- One-hot кодлогдсон хүснэгт (эхний 10 мөр) ---
   BISCUIT  BOURNVITA  BREAD   COCK  COFFEE  CORNFLAKES    JAM  MAGGI   MILK  \
0     True      False   True  False   False       False  False  False   True   
1     True      False   True  False   False        True  False  False   True   
2    False       True   True  False   False       False  False  False  False   
3    False      False   True  False   False       False   True   True   True   
4     True      False  False  False   False       False  False   True  False   
5    False       True   True  False   False       False  False  False  False   
6    False      False  False  False   False        True  False   True  False   
7     True      False   True  False   False       False  False   True  False   
8    False      False   True  False   False       False   True   True  False   
9    False      False   True  False   False       False  False  False   True   

   SUGER    TEA  
0  False  False  
1  False  False  
2  False   Tru

In [33]:
frequent_itemsets = apriori(df_encoded, min_support=0.01, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

In [34]:
sorted_itemsets = frequent_itemsets.sort_values(by='support', ascending=False)
print("\n--- Support-р эрэмбэлсэн багцууд ---")
print(sorted_itemsets)


--- Support-р эрэмбэлсэн багцууд ---
    support                         itemsets  length
2      0.65                          (BREAD)       1
4      0.40                         (COFFEE)       1
0      0.35                        (BISCUIT)       1
10     0.35                            (TEA)       1
5      0.30                     (CORNFLAKES)       1
..      ...                              ...     ...
55     0.05      (MILK, BISCUIT, CORNFLAKES)       3
57     0.05        (BOURNVITA, SUGER, BREAD)       3
17     0.05                 (BISCUIT, SUGER)       2
37     0.05              (CORNFLAKES, MAGGI)       2
82     0.05  (MILK, COFFEE, TEA, CORNFLAKES)       4

[83 rows x 3 columns]


In [35]:
filtered_itemsets = frequent_itemsets[ (frequent_itemsets['length'] == 2) & 
                                       (frequent_itemsets['support'] >= 0.05) ]
print("\n--- 2-itemsets (support >= 0.05) ---")
print(filtered_itemsets)


--- 2-itemsets (support >= 0.05) ---
    support               itemsets  length
11     0.20       (BISCUIT, BREAD)       2
12     0.10        (BISCUIT, COCK)       2
13     0.10      (BISCUIT, COFFEE)       2
14     0.15  (BISCUIT, CORNFLAKES)       2
15     0.10       (BISCUIT, MAGGI)       2
16     0.10        (MILK, BISCUIT)       2
17     0.05       (BISCUIT, SUGER)       2
18     0.10         (BISCUIT, TEA)       2
19     0.15     (BOURNVITA, BREAD)       2
20     0.05    (BOURNVITA, COFFEE)       2
21     0.10     (BOURNVITA, SUGER)       2
22     0.10       (BOURNVITA, TEA)       2
23     0.05          (BREAD, COCK)       2
24     0.15        (BREAD, COFFEE)       2
25     0.05    (BREAD, CORNFLAKES)       2
26     0.10           (BREAD, JAM)       2
27     0.15         (BREAD, MAGGI)       2
28     0.20          (MILK, BREAD)       2
29     0.20         (SUGER, BREAD)       2
30     0.20           (BREAD, TEA)       2
31     0.15         (COFFEE, COCK)       2
32     0.10     

In [36]:
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.7)

In [37]:
final_rules = rules.sort_values("confidence", ascending=False)

print("\n--- Confidence-р эрэмбэлсэн холбоо хамаарлын дүрмүүд ---")
print(final_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])


--- Confidence-р эрэмбэлсэн холбоо хамаарлын дүрмүүд ---
            antecedents           consequents  support  confidence       lift
33  (CORNFLAKES, MAGGI)                 (TEA)     0.05        1.00   2.857143
49      (BISCUIT, COCK)  (COFFEE, CORNFLAKES)     0.10        1.00   5.000000
35          (MILK, JAM)               (MAGGI)     0.05        1.00   4.000000
36        (MILK, MAGGI)                 (JAM)     0.05        1.00  10.000000
37           (TEA, JAM)               (MAGGI)     0.05        1.00   4.000000
..                  ...                   ...      ...         ...        ...
29        (COFFEE, TEA)          (CORNFLAKES)     0.05        1.00   3.333333
30       (MILK, COFFEE)                 (TEA)     0.05        1.00   2.857143
5               (MAGGI)                 (TEA)     0.20        0.80   2.285714
2                (MILK)               (BREAD)     0.20        0.80   1.230769
0           (BOURNVITA)               (BREAD)     0.15        0.75   1.153846

[66 r